## Step 1: Setup and Configuration

In [1]:
!uv pip install unsloth
!uv pip install vllm

Using Python 3.10.19 environment at: /opt/miniconda3/envs/khoina_llamafactory
Audited 1 package in 15ms
Using Python 3.10.19 environment at: /opt/miniconda3/envs/khoina_llamafactory
Audited 1 package in 16ms


In [2]:
%cd /home/khoina/LLaMA-OSS

/home/khoina/LLaMA-OSS


In [3]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
from pathlib import Path

# Set paths
LLAMA_FACTORY_DIR = Path("LLaMA-Factory")
os.chdir(LLAMA_FACTORY_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"LLaMA-Factory directory: {LLAMA_FACTORY_DIR}")
try:
    import unsloth
    from unsloth import FastLanguageModel, is_bfloat16_supported
    print(f"✓ Unsloth imported successfully")
    print(f"  Version: {unsloth.__version__}")
except ImportError as e:
    print("✗ Unsloth not found!")
    print("  Install with: pip install \"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git\"")
    raise e
try:
    from trl import GRPOConfig, GRPOTrainer
    print("✓ TRL imported successfully (GRPO support)")
except ImportError as e:
    print("✗ TRL not found!")
    print("  Install with: pip install trl")
    raise e

print("\nAll dependencies ready for Unsloth GRPO training!")

Working directory: /home/khoina/LLaMA-OSS/LLaMA-Factory
LLaMA-Factory directory: LLaMA-Factory


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/khoina/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-17 23:46:44 [__init__.py:216] Automatically detected platform cuda.


W1117 23:46:44.851000 1010143 site-packages/torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1117 23:46:44.851000 1010143 site-packages/torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
2025-11-17 23:46:44,855 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth imported successfully
  Version: 2025.11.2
✓ TRL imported successfully (GRPO support)

All dependencies ready for Unsloth GRPO training!


## Step 2: Configure Model and Training Parameters

In [ ]:
# Model configuration
MODEL_NAME_OR_PATH = "output/llama3_2_3b_sft_concat_sorted"
MODEL_MAX_LENGTH = 5120

# Training configuration
OUTPUT_DIR = "saves/llama3-3b/grpo_unsloth_3modes"

# Dataset
DATA_DIR = Path("data")
GRPO_LOW_PATH = DATA_DIR / "combined_grpo_train_low.jsonl"
GRPO_MEDIUM_PATH = DATA_DIR / "combined_grpo_train_medium.jsonl"
GRPO_HIGH_PATH = DATA_DIR / "combined_grpo_train_high.jsonl"
VAL_PATH = DATA_DIR / "combined_val.jsonl"
SHUFFLE_DATASET = True

# Performance settings
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
# NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 5.0e-6
MAX_STEPS = 10

# LoRA settings
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# GRPO
NUM_GENERATIONS = 4
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 4096
MAX_PROMPT_LENGTH = MODEL_MAX_LENGTH - MAX_NEW_TOKENS
BETA=0.04,

print("="*70)
print("Configuration Summary")
print("="*70)
print(f"Model: {Path(MODEL_NAME_OR_PATH).name}")
print(f"Output: {OUTPUT_DIR}")
print(f"\nDatasets (3 modes):")
print(f"  • Low:    {GRPO_LOW_PATH.name}")
print(f"  • Medium: {GRPO_MEDIUM_PATH.name}")
print(f"  • High:   {GRPO_HIGH_PATH.name}")
print(f"\nTraining:")
print(f"  • Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  • Gradient accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Effective batch: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Generations/prompt: {NUM_GENERATIONS}")
print("="*70)

Configuration Summary
Model: llama3_2_3b_sft_concat_sorted
Output: saves/llama3-3b/grpo_unsloth_3modes

Datasets (3 modes):
  • Low:    combined_grpo_train_low.jsonl
  • Medium: combined_grpo_train_medium.jsonl
  • High:   combined_grpo_train_high.jsonl

Training:
  • Batch size: 4
  • Gradient accum: 4
  • Effective batch: 16
  • Learning rate: 5e-06
  • Generations/prompt: 4


## Step 3: Load and Combine 3 GRPO Datasets

In [5]:
import json
from datasets import Dataset, concatenate_datasets

def load_jsonl_with_mode(filepath, mode_name):
    """Load JSONL and add mode information."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            if 'mode' not in item:
                item['mode'] = mode_name
            data.append(item)
    return data

print("Loading 3 GRPO datasets...")
print("="*60)

low_data = load_jsonl_with_mode(GRPO_LOW_PATH, "low")
print(f"✓ Low mode:    {len(low_data):,} examples")

medium_data = load_jsonl_with_mode(GRPO_MEDIUM_PATH, "medium")
print(f"✓ Medium mode: {len(medium_data):,} examples")

high_data = load_jsonl_with_mode(GRPO_HIGH_PATH, "high")
print(f"✓ High mode:   {len(high_data):,} examples")

combined_data = low_data + medium_data + high_data
print(f"\n✓ Combined:    {len(combined_data):,} total examples")

train_dataset = Dataset.from_list(combined_data)
print(f"\n✓ Created HuggingFace Dataset with {len(train_dataset)} examples")
print(f"  Columns: {train_dataset.column_names}")

val_data = load_jsonl_with_mode(VAL_PATH, "mixed")
eval_dataset = Dataset.from_list(val_data)
print(f"\n✓ Validation:  {len(eval_dataset):,} examples")

print("\n" + "="*60)
print("Dataset Distribution:")
print("="*60)
mode_counts = {"low": len(low_data), "medium": len(medium_data), "high": len(high_data)}
for mode, count in mode_counts.items():
    pct = 100 * count / len(combined_data)
    print(f"  {mode:8s}: {count:6,} ({pct:5.1f}%)")
print("="*60)

Loading 3 GRPO datasets...
✓ Low mode:    8,897 examples
✓ Medium mode: 9,450 examples
✓ High mode:   8,790 examples

✓ Combined:    27,137 total examples

✓ Created HuggingFace Dataset with 27137 examples
  Columns: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'reasoning', 'combined_index']

✓ Validation:  1,574 examples

Dataset Distribution:
  low     :  8,897 ( 32.8%)
  medium  :  9,450 ( 34.8%)
  high    :  8,790 ( 32.4%)


## Step 4: Load Model with Unsloth + Add LoRA

In [6]:
from unsloth import FastLanguageModel

print("Loading model with Unsloth...")
print("="*60)

# Load SFT model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME_OR_PATH,
    max_seq_length=MODEL_MAX_LENGTH,
    dtype=None,
    load_in_4bit=False,
    gpu_memory_utilization = 0.6,
)

print(f"✓ Loaded model: {Path(MODEL_NAME_OR_PATH).name}")
print(f"✓ Max sequence length: {MODEL_MAX_LENGTH}")
print(f"✓ Device: {next(model.parameters()).device}")
print(f"✓ Dtype: {next(model.parameters()).dtype}")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
    max_seq_length = MODEL_MAX_LENGTH,

)

print(f"\n✓ Added LoRA adapters:")
print(f"  • Rank: {LORA_RANK}")
print(f"  • Alpha: {LORA_ALPHA}")
print(f"  • Dropout: {LORA_DROPOUT}")
print(f"  • Gradient checkpointing: unsloth (optimized)")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
print("="*60)

Loading model with Unsloth...
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.0.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.19it/s]


output/llama3_2_3b_sft_concat_sorted does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
✓ Loaded model: llama3_2_3b_sft_concat_sorted
✓ Max sequence length: 5120
✓ Device: cuda:0
✓ Dtype: torch.bfloat16


Unsloth 2025.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✓ Added LoRA adapters:
  • Rank: 16
  • Alpha: 16
  • Dropout: 0.0
  • Gradient checkpointing: unsloth (optimized)

Trainable parameters: 24,313,856 / 3,237,063,680 (0.75%)


## Step 5: Create Multi-Mode Reward Function for 3 Reasoning Modes

In [7]:
# ============================================================================
# NEW REWARD FUNCTION: 3 Components (3 điểm tối đa)
# ============================================================================

import re
import torch
import math
from typing import Dict, List, Optional, Tuple
from collections import Counter
from scipy import stats
import numpy as np


class ThreeComponentRewardFunction:
    """
    Reward Function với 3 thành phần, mỗi thành phần 1 điểm (tổng 3 điểm):
    
    1. Correct Reward (1đ):
       - có thinking: 0.25
       - extractable (có box): 0.25
       - correct (đúng đáp án): 0.5
    
    2. Length Reward (1đ):
       - Dựa theo hàm CDF của từng mode
       - Normalize về [0,1] để có cùng mức độ điểm
    
    3. Repeat Penalty (1đ):
       - Dùng n-gram repetition * alpha
       - Trừ điểm nếu có repetition cao
    """
    
    def __init__(
        self, 
        mode_statistics: Dict,
        tokenizer=None,
        # Repeat penalty config
        ngram_sizes: Tuple[int] = (3, 4, 5),
        repeat_alpha: float = 2.0,  # Multiplier cho penalty
        repeat_threshold: float = 0.3,  # Ngưỡng để bắt đầu penalty
    ):
        self.mode_statistics = mode_statistics
        self.tokenizer = tokenizer
        self.ngram_sizes = ngram_sizes
        self.repeat_alpha = repeat_alpha
        self.repeat_threshold = repeat_threshold
        self.__name__ = "ThreeComponentRewardFunction"
        
        self._compute_cdf_params()
        
        print("\n" + "="*70)
        print("Three Component Reward Function Initialized")
        print("="*70)
        print("Components:")
        print("  1. Correct Reward (1.0 pts)")
        print("     - Has thinking: 0.25")
        print("     - Extractable box: 0.25")
        print("     - Correct answer: 0.50")
        print("")
        print("  2. Length Reward (1.0 pts)")
        print("     - Based on CDF normalization per mode")
        print("")
        print("  3. Repeat Penalty (1.0 pts)")
        print(f"     - N-gram sizes: {ngram_sizes}")
        print(f"     - Alpha: {repeat_alpha}")
        print(f"     - Threshold: {repeat_threshold}")
        print("="*70)
        
    def _compute_cdf_params(self):
        """Compute CDF parameters for each mode"""
        self.cdf_params = {}
        
        print("\nLength Reward - CDF Parameters:")
        print("-" * 60)
        
        for mode, stats in self.mode_statistics.items():
            mean = stats['mean']
            std = stats['std']
            
            self.cdf_params[mode] = {'mean': mean, 'std': std}
            print(f"{mode.upper():8s}: mean={mean:.1f}, std={std:.1f}")
    
    # =========================================================================
    # REWARD 1: CORRECTNESS (1 điểm)
    # =========================================================================
    
    def has_thinking_tags(self, text: str) -> bool:
        """Check if text has <think>...</think> tags"""
        return bool(re.search(r'<think>.*?</think>', text, re.DOTALL))
    
    def extract_answer(self, text: str) -> Optional[str]:
        """Extract answer from \\boxed{} or \\box{}"""
        matches = re.findall(r'\\boxed\{([^}]+)\}', text)
        if matches:
            return matches[-1].strip()
        
        matches = re.findall(r'\\box\{([^}]+)\}', text)
        if matches:
            return matches[-1].strip()
        
        return None
    
    def normalize_answer(self, answer: str) -> str:
        """Normalize answer for comparison"""
        answer = answer.lower().strip()
        answer = answer.replace(" ", "").replace(",", "")
        answer = answer.replace("\\%", "").replace("%", "")
        answer = answer.replace("^\\circ", "").replace("°", "")
        answer = answer.replace("\\$", "").replace("$", "")
        answer = answer.replace("\\pi", "pi")
        
        if '/' in answer and '\\' not in answer:
            try:
                parts = answer.split('/')
                answer = str(float(parts[0]) / float(parts[1]))
            except:
                pass
        
        return answer
    
    def check_correctness(self, prediction: str, ground_truth: str) -> bool:
        """Check if prediction matches ground truth"""
        if not ground_truth or not prediction:
            return False
        
        pred_norm = self.normalize_answer(prediction)
        truth_norm = self.normalize_answer(ground_truth)
        
        if pred_norm == truth_norm:
            return True
        
        try:
            val_pred = float(pred_norm)
            val_truth = float(truth_norm)
            rel_error = abs(val_pred - val_truth) / max(abs(val_truth), 1e-10)
            if rel_error < 0.001:
                return True
        except:
            pass
        
        return False
    
    def compute_correctness_reward(self, completion: str, ground_truth: Optional[str] = None) -> float:
        """
        Compute Reward 1: Correctness (max 1.0 điểm)
        """
        reward = 0.0
        
        # Component 1.1: Has thinking tags (0.25)
        if self.has_thinking_tags(completion):
            reward += 0.25
        
        # Component 1.2: Extractable answer (0.25)
        extracted = self.extract_answer(completion)
        if extracted is not None:
            reward += 0.25
        
        # Component 1.3: Correct answer (0.5)
        if ground_truth is not None and extracted is not None:
            if self.check_correctness(extracted, ground_truth):
                reward += 0.5
        
        return reward
    
    # =========================================================================
    # REWARD 2: LENGTH (1 điểm) - CDF based
    # =========================================================================
    
    def count_reasoning_tokens(self, text: str) -> int:
        """Count tokens inside <think>...</think>"""
        matches = re.findall(r'<think>(.*?)</think>', text, re.DOTALL)
        if not matches:
            return 0
        
        reasoning_text = " ".join(matches)
        
        if self.tokenizer is not None:
            tokens = self.tokenizer(reasoning_text, add_special_tokens=False)
            return len(tokens['input_ids'])
        else:
            return int(len(reasoning_text.split()) * 1.3)
    
    def compute_length_reward_cdf(self, tokens: int, mode: str) -> float:
        """
        Compute Reward 2: Length reward using CDF normalization (max 1.0 điểm)
        
        Idea: 
        - Sử dụng CDF của phân phối Gaussian (mean, std của mode)
        - Token gần mean → CDF gần 0.5 → reward cao
        - Token xa mean → CDF gần 0 hoặc 1 → reward thấp
        """
        if mode not in self.cdf_params:
            return 0.0
        
        params = self.cdf_params[mode]
        mean = params['mean']
        std = params['std']
        
        if tokens == 0:
            return 0.0
        
        # Compute CDF value
        z_score = (tokens - mean) / std
        cdf_value = stats.norm.cdf(z_score)
        
        # Transform: reward = 1 - 2*|CDF - 0.5|
        # CDF = 0.5 (at mean) → reward = 1.0
        # CDF = 0 or 1 (far from mean) → reward = 0.0
        reward = 1.0 - 2.0 * abs(cdf_value - 0.5)
        reward = max(0.0, min(1.0, reward))
        
        return reward
    
    # =========================================================================
    # REWARD 3: REPEAT PENALTY (1 điểm)
    # =========================================================================
    
    def compute_ngram_repetition(self, text: str, n: int) -> float:
        """Compute n-gram repetition ratio"""
        tokens = text.split()
        if len(tokens) < n:
            return 0.0
        
        ngrams = [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        if len(ngrams) == 0:
            return 0.0
        
        ngram_counts = Counter(ngrams)
        repeated = sum(count - 1 for count in ngram_counts.values() if count > 1)
        
        return repeated / len(ngrams)
    
    def compute_max_repetition(self, text: str) -> float:
        """Compute maximum repetition across all n-gram sizes"""
        max_rep = 0.0
        
        for n in self.ngram_sizes:
            rep = self.compute_ngram_repetition(text, n)
            max_rep = max(max_rep, rep)
        
        return max_rep
    
    def compute_repeat_penalty(self, completion: str) -> float:
        """
        Compute Reward 3: Repeat penalty (max 1.0 điểm penalty)
        """
        matches = re.findall(r'<think>(.*?)</think>', completion, re.DOTALL)
        if not matches:
            return 0.0
        
        thinking_text = " ".join(matches)
        max_rep = self.compute_max_repetition(thinking_text)
        
        if max_rep <= self.repeat_threshold:
            return 0.0
        
        # Scale penalty
        scaled_rep = (max_rep - self.repeat_threshold) / (1.0 - self.repeat_threshold)
        penalty = self.repeat_alpha * scaled_rep
        penalty = min(1.0, penalty)
        
        return penalty
    
    # =========================================================================
    # MAIN REWARD COMPUTATION
    # =========================================================================
    
    def detect_mode_from_prompt(self, prompt: str) -> str:
        """Extract mode from prompt: <mode: low|medium|high>"""
        match = re.search(r'<mode:\s*(low|medium|high)>', prompt)
        return match.group(1) if match else "medium"
    
    def __call__(
        self, 
        prompts: List[str], 
        completions: List[str], 
        modes: Optional[List[str]] = None,
        ground_truths: Optional[List[str]] = None,
        **kwargs
    ) -> torch.Tensor:
        """
        Compute combined reward: R_total = R1 + R2 - R3
        
        Range: [-1, 2] (typically [0, 2])
        """
        rewards = []
        
        for i, completion in enumerate(completions):
            # Detect mode
            if modes is not None and i < len(modes):
                mode = modes[i]
            else:
                mode = self.detect_mode_from_prompt(prompts[i])
            
            # Get ground truth
            gt = ground_truths[i] if ground_truths is not None and i < len(ground_truths) else None
            
            # Reward 1: Correctness (0-1)
            r1_correct = self.compute_correctness_reward(completion, gt)
            
            # Reward 2: Length (0-1)
            tokens = self.count_reasoning_tokens(completion)
            r2_length = self.compute_length_reward_cdf(tokens, mode)
            
            # Reward 3: Repeat Penalty (0-1)
            r3_penalty = self.compute_repeat_penalty(completion)
            
            # Combined: R1 + R2 - R3
            total_reward = r1_correct + r2_length - r3_penalty
            
            rewards.append(total_reward)
        
        return torch.tensor(rewards, dtype=torch.float32)
    
    def compute_reward_components(
        self,
        prompt: str,
        completion: str,
        mode: str,
        ground_truth: Optional[str] = None
    ) -> Dict[str, float]:
        """Compute all reward components separately for analysis"""
        r1 = self.compute_correctness_reward(completion, ground_truth)
        
        tokens = self.count_reasoning_tokens(completion)
        r2 = self.compute_length_reward_cdf(tokens, mode)
        
        r3 = self.compute_repeat_penalty(completion)
        
        total = r1 + r2 - r3
        
        return {
            'r1_correct': r1,
            'r2_length': r2,
            'r3_penalty': r3,
            'total': total,
            'tokens': tokens
        }

In [8]:
# ============================================================================
# Compute Mode Statistics từ Training Data
# ============================================================================

def compute_mode_statistics_from_dataset(dataset):
    """Compute mean, std for each mode"""
    from collections import defaultdict
    import numpy as np
    
    mode_tokens = defaultdict(list)
    
    for example in dataset:
        mode = example['mode']
        tokens = example.get('reasoning_tokens', 0)
        if tokens > 0:
            mode_tokens[mode].append(tokens)
    
    mode_stats = {}
    for mode, tokens_list in mode_tokens.items():
        tokens_array = np.array(tokens_list)
        mode_stats[mode] = {
            'mean': float(np.mean(tokens_array)),
            'std': float(np.std(tokens_array)),
            'min': float(np.min(tokens_array)),
            'max': float(np.max(tokens_array)),
            'p25': float(np.percentile(tokens_array, 25)),
            'p50': float(np.percentile(tokens_array, 50)),
            'p75': float(np.percentile(tokens_array, 75)),
            'count': len(tokens_list)
        }
    
    print("\nMode Statistics:")
    print("="*60)
    for mode in ['low', 'medium', 'high']:
        if mode in mode_stats:
            s = mode_stats[mode]
            print(f"{mode.upper():8s}: mean={s['mean']:.1f}, std={s['std']:.1f}, "
                  f"n={s['count']}")
    print("="*60)
    
    return mode_stats

# Compute statistics
mode_stats = compute_mode_statistics_from_dataset(train_dataset)

reward_function = ThreeComponentRewardFunction(
    mode_statistics=mode_stats,
    tokenizer=tokenizer,
    ngram_sizes=(3, 4, 5),
    repeat_alpha=2.0,  # Có thể điều chỉnh (1.0 - 3.0)
    repeat_threshold=0.3  # Có thể điều chỉnh (0.2 - 0.4)
)


Mode Statistics:
LOW     : mean=128.0, std=106.7, n=8897
MEDIUM  : mean=483.2, std=448.9, n=9450
HIGH    : mean=1115.5, std=1039.7, n=8790

Length Reward - CDF Parameters:
------------------------------------------------------------
LOW     : mean=128.0, std=106.7
MEDIUM  : mean=483.2, std=448.9
HIGH    : mean=1115.5, std=1039.7

Three Component Reward Function Initialized
Components:
  1. Correct Reward (1.0 pts)
     - Has thinking: 0.25
     - Extractable box: 0.25
     - Correct answer: 0.50

  2. Length Reward (1.0 pts)
     - Based on CDF normalization per mode

  3. Repeat Penalty (1.0 pts)
     - N-gram sizes: (3, 4, 5)
     - Alpha: 2.0
     - Threshold: 0.3


In [9]:
# p = "<think></think>"
# tokens = tokenizer(p, add_special_tokens=False)
# len(tokens['input_ids'])

## Step 6: Configure and Run GRPO Training with Unsloth

In [10]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
from transformers import GenerationConfig
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    # num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_NEW_TOKENS,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    num_generations=NUM_GENERATIONS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    logging_steps=1,
    save_steps=100,
    max_grad_norm=1.0,
    save_total_limit=2,
    report_to="none",
    seed=42,
    dataloader_num_workers=4,
    shuffle_dataset=SHUFFLE_DATASET,
    # use_vllm=True,
)

print("="*70)
print("GRPO Training Configuration")
print("="*70)
print(f"Output directory: {OUTPUT_DIR}")
# print(f"Epochs: {NUM_TRAIN_EPOCHS}")
print(f"Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Generations per prompt: {NUM_GENERATIONS}")
print(f"Temperature: {TEMPERATURE}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print("="*70)

# Prepare dataset formatting
def format_prompt(example):
    """Format example for GRPO training."""
    mode = example.get("mode", "medium")
    question = f"Question: {example['prompt']}"
    instruction_prompt = "Please reason step by step, and put your final answer within \\boxed{}. Do NOT include explanations or reasoning in the final answer - only the mathematical expression or value in \\boxed{}."
    if mode == "low":
        mode_instruction = "Fast response required. Minimize reasoning time. Direct answers only.\n"
        prompt = mode_instruction + question + " " + instruction_prompt        
    elif mode == "medium":
        mode_instruction = f"Balanced approach. Think enough but do not overthink. Reasonable depth.\n"
        prompt = mode_instruction + question + " " + instruction_prompt
    else:  # mode == "high"
        mode_instruction = f"Maximum accuracy focus. Use extensive reasoning. Verify thoroughly.\n"
        prompt = mode_instruction + question + " " + instruction_prompt
    
    return {
        "prompt": prompt,
        "chosen": example.get("response", ""),
        "mode": mode,
    }

# Format datasets
train_dataset_formatted = train_dataset.map(format_prompt, remove_columns=train_dataset.column_names)
eval_dataset_formatted = eval_dataset.map(format_prompt, remove_columns=eval_dataset.column_names)

print(f"\n✓ Formatted {len(train_dataset_formatted)} training examples")
print(f"✓ Formatted {len(eval_dataset_formatted)} validation examples")
print(f"\nDataset columns: {train_dataset_formatted.column_names}")

GRPO Training Configuration
Output directory: saves/llama3-3b/grpo_unsloth_3modes
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 5e-06
Generations per prompt: 4
Temperature: 0.7
Max new tokens: 4096


Map: 100%|██████████| 1574/1574 [00:00<00:00, 15914.71 examples/s]


✓ Formatted 27137 training examples
✓ Formatted 1574 validation examples

Dataset columns: ['prompt', 'mode', 'chosen']


In [ ]:
print("\nInitializing GRPO Trainer...")

reward_function.__name__ = "custom_multimode_reward"
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    tokenizer=tokenizer,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    reward_funcs=[reward_function],
    max_length=MODEL_MAX_LENGTH,
)


print("✓ GRPO Trainer initialized")
print("\n" + "="*70)
print("Starting GRPO Training with 3 Reasoning Modes!")
print("="*70)
print(f"\nTraining {len(train_dataset_formatted):,} examples")
print(f"  • Low mode:    {len(low_data):,}")
print(f"  • Medium mode: {len(medium_data):,}")
print(f"  • High mode:   {len(high_data):,}")
print(f"\nGenerating {NUM_GENERATIONS} responses per prompt")
print(f"Learning which responses match target reasoning depth\n")
print("="*70)

# Start training
trainer.train()

print("\n" + "="*70)
print("GRPO Training Completed!")
print("="*70)
print(f"Model saved to: {OUTPUT_DIR}")

The model is already on multiple devices. Skipping the move to device specified in `args`.



Initializing GRPO Trainer...
✓ GRPO Trainer initialized

Starting GRPO Training with 3 Reasoning Modes!

Training 27,137 examples
  • Low mode:    8,897
  • Medium mode: 9,450
  • High mode:   8,790

Generating 4 responses per prompt
Learning which responses match target reasoning depth



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,137 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072}. If this is not desired, please set these values explicitly.


In [ ]:
model.save_lora("grpo_saved_lora")

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [ ]:
messages = [
    {"role": "system", "content": "Fast response required. Minimize reasoning time. Direct answers only."},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

## Step 7: Save the Trained Model

In [ ]:
print("Saving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")